### Load Data

In [1]:
import pandas as pd
df1=pd.read_csv("cleaned_reviews_all_flags_scored.csv")
df2=pd.read_csv("cleaned_reviews_no_lemma_no_stopwords_scored.csv")
df3=pd.read_csv("cleaned_reviews_no_speical_no_lowercase_scored.csv")

### Apply Bag of Words

In [2]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer1 = CountVectorizer()

bow_matrix1 = vectorizer1.fit_transform(df1["review_body"])

bow_df1 = pd.DataFrame(
    bow_matrix1.toarray(),
    columns=vectorizer1.get_feature_names_out()
)

df_bow1 = pd.concat([df1.reset_index(drop=True), bow_df1], axis=1)

In [3]:
vectorizer2 = CountVectorizer()

bow_matrix2 = vectorizer2.fit_transform(df2["review_body"])

bow_df2 = pd.DataFrame(
    bow_matrix2.toarray(),
    columns=vectorizer2.get_feature_names_out()
)

df_bow2 = pd.concat([df2.reset_index(drop=True), bow_df2], axis=1)

In [4]:
vectorizer3 = CountVectorizer()

bow_matrix3 = vectorizer3.fit_transform(df3["review_body"])

bow_df3 = pd.DataFrame(
    bow_matrix3.toarray(),
    columns=vectorizer3.get_feature_names_out()
)

df_bow3 = pd.concat([df3.reset_index(drop=True), bow_df3], axis=1)

### Save Data

In [5]:
df_bow1.to_csv("bow_all_flags.csv", index=False)
df_bow2.to_csv("bow_no_lemma_no_stopwords.csv", index=False)
df_bow3.to_csv("bow_no_speical_no_lowercase.csv", index=False)

### Apply GLoVe

In [6]:
import numpy as np

glove_path = "glove.6B.100d.txt"

embeddings_index = {}

with open(glove_path, encoding="utf8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        vec = np.asarray(values[1:], dtype="float32")
        embeddings_index[word] = vec

In [7]:
def get_sentence_vector(text, dim=100):
    words = str(text).split()
    vectors = [embeddings_index[w] for w in words if w in embeddings_index]
    
    if len(vectors) == 0:
        return np.zeros(dim)
    
    return np.mean(vectors, axis=0)

df1["glove_vector"] = df1["review_body"].apply(get_sentence_vector)
df2["glove_vector"] = df2["review_body"].apply(get_sentence_vector)
df3["glove_vector"] = df3["review_body"].apply(get_sentence_vector)

In [8]:
glove_df1 = pd.DataFrame(df1["glove_vector"].tolist())
glove_df2 = pd.DataFrame(df2["glove_vector"].tolist())
glove_df3 = pd.DataFrame(df3["glove_vector"].tolist())

In [9]:
df1_final = pd.concat([df1.reset_index(drop=True), glove_df1], axis=1)
df1_final.to_csv("glove_all_flags.csv", index=False)
df2_final = pd.concat([df2.reset_index(drop=True), glove_df2], axis=1)
df2_final.to_csv("glove_no_lemma_no_stopwords.csv", index=False)
df3_final = pd.concat([df3.reset_index(drop=True), glove_df3], axis=1)
df3_final.to_csv("glove_no_special_no_lowercase.csv", index=False)